In [ ]:
import pandas as pd
import pyterrier as pt
if not pt.started():
  pt.init()
indexref = pt.IndexRef.of("/path/to/data.properties")

In [ ]:
topics.iloc[55]

In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics2020.xml')
data = tree.getroot()
titles = []
title_ids = []
nums = []
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics.xml')
data = tree.getroot()    
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
topics = pd.DataFrame()
topics['qid'] = nums
topics['query'] = titles

qrels_path=("/workspace/src/evaluate_targers/qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)


In [ ]:
path_targer = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/targer_indexes/'
indexref_orig = pt.IndexRef.of(path_targer + "pd_index_original-100/data.properties")

index = pt.IndexFactory.of(indexref_orig)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res_orig=retr.transform(topics)

res_orig

In [ ]:
eval_orig = pt.Utils.evaluate(res_orig,qrels,metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"])
eval_orig

In [ ]:
!ls /mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/DeepCT_indexes/pools_topics_conclusionspd_index

In [ ]:
path_deepct = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/DeepCT_indexes/pools_topics_conclusions'
indexref_deepct = pt.IndexRef.of(path_deepct + "pd_index/data.properties")

index = pt.IndexFactory.of(indexref_deepct)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res_deepct=retr.transform(topics)

res_deepct

In [ ]:
eval_deepct = pt.Utils.evaluate(res_deepct,qrels,metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"])
eval_deepct

In [ ]:
res_deepct.groupby(['qid'])

In [ ]:
docnos = qrels.docno
docnos_in_orig = [res_orig['docno'].head(25).isin(docnos)]
docnos_in_orig

In [ ]:
docnos_in_deepct = res_deepct['docno'].head(25).isin(docnos).to_list()
# docnos_in_deepct

In [ ]:
res_orig.iloc[17]

In [ ]:
res_deepct.iloc[8]

In [ ]:
for i in range(100):
#     print(i)
    if True in (res_deepct[res_deepct['qid'] == str(i)]['docno']).head(25).isin(docnos).to_list():
        print(i)
#         print((res_orig[res_orig['qid'] == str(i)]['docno'].head(25)).isin(docnos))
        

In [ ]:
for docno in (res_deepct[res_deepct['qid'] == str(58)]['docno']).head(25).to_list():
    print(qrels[qrels['docno'] == docno]['label'])

In [ ]:
for docno in (res_orig[res_orig['qid'] == str(58)]['docno']).head(25).to_list():
    print(qrels[qrels['docno'] == docno]['label'])

In [ ]:
topics.iloc[56]['query']

In [ ]:
(res_deepct[res_deepct['qid'] == str(58)]['docno']).head(25).isin(docnos).to_list()

In [ ]:
(res_orig[res_orig['qid'] == str(58)]['docno']).head(25).isin(docnos).to_list()

In [ ]:
qrels[qrels['qid'] == str(58)]

In [ ]:
(res_deepct[res_deepct['qid'] == str(58)]['docno']).to_list().index('Sce1fe489-Aa0395e1d')

In [ ]:
(res_orig[res_orig['qid'] == str(58)]['docno']).to_list().index('Sce1fe489-Aa0395e1d')

In [ ]:
(res_deepct[res_deepct['qid'] == str(58)]['docno']).head(7)

In [ ]:
top25_deepct = (res_deepct[res_deepct['qid'] == str(58)]['docno']).head(25).to_list()
top25_orig = (res_orig[res_orig['qid'] == str(58)]['docno']).head(25).to_list()

missed_by_orig = []

for docno in docnos:
    if docno in top25_deepct and docno not in top25_orig:
        missed_by_orig.append(docno)
        print(docno)

In [ ]:
top25_deepct = (res_deepct[res_deepct['qid'] == str(58)]['docno']).head(25).to_list()
top25_orig = (res_orig[res_orig['qid'] == str(58)]['docno']).head(25).to_list()

missed = []

for docno in docnos:
    if docno in top25_orig and docno not in top25_deepct:
        missed.append(docno)
        print(docno)

In [ ]:
print([docno for docno in docnos if docno in top25_deepct])
print([docno for docno in docnos if docno in top25_orig])

In [ ]:
founded = []

for docno in docnos:
    if docno not in top25_orig and docno in top25_deepct:
        founded.append(docno)
        print(docno)

In [ ]:
from tqdm import tqdm
import ir_datasets
dataset = ir_datasets.load('argsme/2020-04-01')
for doc in tqdm(dataset.docs_iter(), total=387740):

    if doc.doc_id in missed:
        print(f'missed : {doc.source_url, doc.premises_texts}')
    elif doc.doc_id in founded:
          print(f'founded : {doc.source_url, doc.premises_texts}')

In [ ]:
#docnos that are retrieved both by original dataset and deepct but not considered relevant 
overlap = [docno for docno in top25_orig if ((docno in top25_deepct) and (docno not in docnos))]
print(f'number of documents both retrieved by deepct and original dataset, but not considered relevant : {len(overlap)} from 25')

overlap

In [ ]:
first_unrelevant_deepct = 'S6fc99c30-Ae718e9a8'
for doc in tqdm(dataset.docs_iter(), total=387740):
    if doc.doc_id == first_unrelevant_deepct:
        print(f'first_unrelevant_deepct : {doc.source_url, doc.premises_texts}')

In [ ]:
arg_relevance_one = 'Sce1fe489-Aa0395e1d'
for doc in tqdm(dataset.docs_iter(), total=387740):
    if doc.doc_id == arg_relevance_one:
        print(f'arg_relevance_one : {doc.source_url, doc.premises_texts}')